# GIADA42 — GPU-only recovery
Restore saved60 native cases and15 passive checks. Fix seed-axis adapter; complete original GPU evaluations only. No native phase, teacher checkout or NEURON installation.41r and original thresholds unchanged.

In [ ]:
import hashlib,json,os,shutil,subprocess,sys,zipfile,time,threading
from pathlib import Path
REVISION='PINNED_BY_CONSOLE_DISPATCH'
SOURCE_SHA='7e07984c6c8770fee223a83f4694922c2f8cb92529e3df899bc511cf8d5080be'
WORK=Path('/tmp/giada_task42_transport_recovery');WORK.mkdir(exist_ok=False)
repo=WORK/'giada';out=WORK/'result';public=Path('/kaggle/working');log=public/'process.log'
code=None;started=time.time()
def setup(cmd):
 with log.open('a') as stream:subprocess.run(cmd,stdout=stream,stderr=subprocess.STDOUT,check=True)
try:
 assert len(REVISION)==40,'Use console dispatch to pin the exact commit'
 native_candidates=sorted(Path('/kaggle/input').rglob('native42.zip.bin'))
 native_verified=[p for p in native_candidates if p.is_file() and p.stat().st_size==18226046 and hashlib.sha256(p.read_bytes()).hexdigest()=='9ef80d5e3c4f4188c1f7f6e5ac41d7b7bafda409fe6ba3d6efabb5206c0d7786']
 assert native_verified,'Saved42 input missing/hash mismatch: GPU worker not started; inspect mounted input dataset'
 native_archive=native_verified[0]
 (public/'input_transport_receipt.json').write_text(json.dumps(dict(valid=True,mounted_path=str(native_archive),bytes=native_archive.stat().st_size,sha256='9ef80d5e3c4f4188c1f7f6e5ac41d7b7bafda409fe6ba3d6efabb5206c0d7786',native_regenerated=False,worker_started=False)))
 print('[GIADA42 recovery] mounted saved archive SHA256 verified before GPU worker',flush=True)
 print('[GIADA42 recovery] pinned Torch/GPU setup',flush=True)
 setup([sys.executable,'-m','pip','install','--quiet','--upgrade','torch==2.11.0+cu130','--index-url','https://download.pytorch.org/whl/cu130'])
 for cmd in (['git','clone','--quiet','--depth','1','--branch','codex/surrogate-validity-audit','https://github.com/Zagred47/giada.git',str(repo)],['git','-C',str(repo),'fetch','--quiet','--depth','1','origin',REVISION],['git','-C',str(repo),'checkout','--quiet','--detach','FETCH_HEAD']):setup(cmd)
 sources=list(Path('/kaggle/input').rglob('source41n.zip.bin'));assert len(sources)==1
 with sources[0].open('rb') as f:assert hashlib.file_digest(f,'sha256').hexdigest()==SOURCE_SHA
 env=dict(os.environ,CUBLAS_WORKSPACE_CONFIG=':4096:8',MPLBACKEND='Agg',GIADA_TASK42_NATIVE_ARCHIVE=str(native_archive))
 with log.open('a') as f:
  proc=subprocess.Popen([sys.executable,'-u',str(repo/'scripts/run_task42_gpu_recovery.py'),str(sources[0]),str(out)],cwd=repo,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
  timer=threading.Timer(5400,proc.terminate);timer.daemon=True;timer.start()
  for line in proc.stdout:
   f.write(line);f.flush()
   if line.startswith('[GIADA') or 'Error' in line or 'Traceback' in line:print(line.rstrip(),flush=True)
  code=proc.wait();timer.cancel()
finally:
 out.mkdir(exist_ok=True)
 if log.exists():shutil.copy2(log,out/'process.log')
 (out/'runtime_receipt.json').write_text(json.dumps(dict(revision=REVISION,source_sha256=SOURCE_SHA,worker_exit_code=code,elapsed_seconds=time.time()-started,training_updates_performed=0,selection_recomputed=False,thresholds_changed=False)))
 archive=Path(shutil.make_archive(str(public/'task42_transport_final'),'zip',out))
 with archive.open('rb') as f:digest=hashlib.file_digest(f,'sha256').hexdigest()
 (public/'kaggle_receipt.json').write_text(json.dumps(dict(valid=code==0,revision=REVISION,source_sha256=SOURCE_SHA,archive=archive.name,sha256=digest,bytes=archive.stat().st_size,worker_exit_code=code,training_updates_performed=0,selection_recomputed=False,thresholds_changed=False)))
 print('[GIADA42 recovery] final archive persisted '+digest,flush=True)
assert code==0,'Evaluation recovery failed technically; preserve output and logs'
